# Download, aggregate, and visualize filtered activations

This notebook selects completion records by prompt metadata, downloads their cached activations from GCS, aggregates cached token positions, projects the resulting feature vectors with PCA, and visualizes the projections.

The workflow is organized into six stages: setup, configuration, data selection and download, activation aggregation, PCA projection, and visualization.

## 1. Setup

### 1.1 Optional Colab bootstrap

Uncomment and run the next cell only when starting from a fresh Colab runtime. Local runs can skip it.

In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
%cd temporal-manifolds
# !gcloud auth application-default login
# !mv -n .env.example .env

### 1.2 Imports and repository paths

Resolve the repository root and import the activation download, filtering, and aggregation utilities.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from temporal_manifolds.activations.extract_activations import load_selected_node_groups
from temporal_manifolds.utils.activation_aggregation import (
    aggregate_activation_file,
    nodes_for_classes,
)
from temporal_manifolds.utils.completion_filters import (
    download_activation_files,
    find_activation_paths,
)

## 2. Configure the workflow

Edit the next cell before running the download and analysis stages.

### Selection and download controls

- `PROMPT_FRAMING`, `OUTPUT_FORMAT`, and `TASK_METADATA_FILTERS` select completion records. Use `None` to disable a filter; every entry in `TASK_METADATA_FILTERS` must match.
- `MAX_FILES` caps the selected activation files, while `OVERWRITE` controls whether existing local files are replaced.

### Activation controls

- `NODE_CLASSES` unions selected-node classes; `None` includes every available class.
- `RESIDUAL_STREAM_LAYERS` uses `None` for every layer, a set of layer numbers for a subset, or `set()` to exclude residual streams.
- `AGGREGATION_POLICY` chooses the first assistant position (`"assistant"`) or the mean over all cached positions (`"all"`).

### Interactive visualization

The visualization section discovers scalar prompt-metadata fields automatically. Use its controls to choose the color field, optionally choose a filter field, and select the values to keep.

In [ ]:
PROMPT_FRAMING: str | None = "task_available_time"
OUTPUT_FORMAT: str | None = None
TASK_METADATA_FILTERS: dict[str, object] | None = {
    "domain": "communication",
}
COMPLETIONS_PATH = repo_root / "data" / "completions_256.jsonl"
COMPLETIONS_GCS_BUCKET = "temporal-research-bucket"
COMPLETIONS_GCS_PREFIX = "completions"
ACTIVATIONS_DIR = repo_root / "results" / "feature_geometry_after_assistant_residual_stream"
SELECTED_NODES_PATH = repo_root / "data" / "selected_nodes" / "final_500_eap_ig.pkl"
# SELECTED_NODES_PATH = repo_root / "eap-ig-selected-100"/"data"/"selected_nodes"/"eap_ig_selected_100"/"final_500_eap_ig.pkl"
# SELECTED_NODES_PATH = repo_root / "eap-ig-selected-100"/"data"/"selected_nodes"/"eap_ig_selected_100"/"final_node_list.pkl"
NODE_CLASSES: set[str] | None = {"p_generic", "n_generic"}
RESIDUAL_STREAM_LAYERS: set[int] | None = set()
AGGREGATION_POLICY = "assistant"  # assistant or all
MAX_FILES: int | None = 1000000
OVERWRITE = False

## 3. Select and download data

### 3.1 Download the completions index

Download `gs://temporal-research-bucket/completions/completions_256.jsonl` through the authenticated Google Cloud Storage API before filtering it. An existing local file is skipped unless `OVERWRITE` is `True`.

In [ ]:
download_activation_files(
    [COMPLETIONS_PATH],
    gcs_prefix=COMPLETIONS_GCS_PREFIX,
    overwrite=OVERWRITE,
    upload_root=COMPLETIONS_PATH.parent,
    bucket_name=COMPLETIONS_GCS_BUCKET,
)
print(f"Completions file: {COMPLETIONS_PATH}")

### 3.2 Filter completion records

Find activation paths whose completion metadata matches the configured filters, apply `MAX_FILES`, and preview the selected paths.

In [ ]:
matching_paths = find_activation_paths(
    prompt_framing=PROMPT_FRAMING,
    output_format=OUTPUT_FORMAT,
    task_metadata=TASK_METADATA_FILTERS,
    completions_path=COMPLETIONS_PATH,
    activations_dir=ACTIVATIONS_DIR,
)
paths_to_download = matching_paths if MAX_FILES is None else matching_paths[:MAX_FILES]

print(f"Found {len(matching_paths):,} matching activation files.")
print(f"Selected {len(paths_to_download):,} files for download.")
for path in paths_to_download[:10]:
    print(path)

### 3.3 Download activation caches and node definitions

Existing local files are skipped unless `OVERWRITE` is `True`. Activation files are downloaded from the hardcoded `conversational_after_assistant_residual_stream/results/feature_geometry_after_assistant_residual_stream` GCS path. The selected-node definitions use the repository-relative GCS convention.

In [ ]:
downloaded_paths = download_activation_files(
    paths_to_download,
    gcs_prefix=(
        "conversational_after_assistant_residual_stream/"
        "results/feature_geometry_after_assistant_residual_stream"
    ),
    overwrite=OVERWRITE,
    upload_root=ACTIVATIONS_DIR,
    bucket_name="temporal-research-bucket",
)
download_activation_files(
    [SELECTED_NODES_PATH],
    gcs_prefix="eap-ig/data/selected_nodes",
    upload_root=SELECTED_NODES_PATH.parent,
    bucket_name="temporal-research-bucket",
)

print(f"Download complete: {len(downloaded_paths):,} activation paths.")
print(f"Selected-node definitions: {SELECTED_NODES_PATH}")

## 4. Aggregate cached activations

Cached tensors begin as `batch x cached positions x features`; selected attention tensors additionally retain their head-feature dimension. The same policy is applied to every included MLP, attention, and complete residual-stream tensor:

- `assistant`: keep the first cached position.
- `all`: average every cached token position.

In [ ]:
if not downloaded_paths:
    raise ValueError("No activation files were selected.")

selected_node_groups = load_selected_node_groups(SELECTED_NODES_PATH)
print("Available node classes:", sorted(selected_node_groups))
allowed_nodes = nodes_for_classes(selected_node_groups, NODE_CLASSES)
print("Node-class filter:", "all" if NODE_CLASSES is None else sorted(NODE_CLASSES))
print(
    "Residual-stream layers:",
    "all" if RESIDUAL_STREAM_LAYERS is None else sorted(RESIDUAL_STREAM_LAYERS),
)

aggregated_by_file = []
for path in downloaded_paths:
    aggregated_by_file.append(
        aggregate_activation_file(
            path,
            AGGREGATION_POLICY,
            allowed_nodes,
            RESIDUAL_STREAM_LAYERS,
        )
    )

print(f"Aggregated {len(aggregated_by_file):,} files with policy={AGGREGATION_POLICY!r}.")
for activation_type, tensors in aggregated_by_file[0]["activations"].items():
    example_shapes = {name: tuple(tensor.shape) for name, tensor in list(tensors.items())[:3]}
    print(activation_type, example_shapes)
print("Retained selected nodes:", {
    name: len(indices)
    for name, indices in aggregated_by_file[0]["node_indices"].items()
})
print("Included residual streams:", aggregated_by_file[0]["included_residual_streams"])

## 5. Build the feature matrix and project with PCA

### 5.1 Flatten activations into one vector per sample

Concatenate the aggregated MLP vectors and flattened attention-head tensors in a stable iteration order, then stack the samples into a single matrix.

In [ ]:
import torch

feature_vectors = []
for sample in aggregated_by_file:
    sample_features = list(sample["activations"]["mlp"].values())
    sample_features.extend(
        attention.ravel()
        for attention in sample["activations"]["attn"].values()
    )
    feature_vectors.append(torch.cat(sample_features, dim=0))

activation_matrix = torch.stack(feature_vectors).to(torch.float)
print("Activation matrix shape:", tuple(activation_matrix.shape))

### 5.2 Compute the first three principal components

Fit a low-rank PCA basis and center the activation matrix before projecting each sample into three dimensions.

In [ ]:
_, _, principal_directions = torch.pca_lowrank(activation_matrix)
projection_directions = principal_directions[:, :3]
projs = (activation_matrix - activation_matrix.mean(dim=0, keepdim=True)) @ projection_directions
print("Projection shape:", tuple(projs.shape))

## 6. Visualize the PCA projections

Load the selected samples into one DataFrame, then explore their PCA projections with interactive color and filter controls. Nested metadata keys appear as dotted paths such as `task_metadata.difficulty`.

### 6.1 Prepare projections and metadata

Flatten scalar prompt metadata, add the derived time-horizon fields, and align every row with its PCA projection.

In [ ]:
import json

import numpy as np
import pandas as pd

projs_np = projs.detach().cpu().numpy()
sample_indices = [
    int(path.stem.removeprefix("activations_sample_"))
    for path in downloaded_paths
]
target_indices = set(sample_indices)
completion_metadata = {}
with COMPLETIONS_PATH.open(encoding="utf-8") as completion_file:
    for index, line in enumerate(completion_file):
        if index in target_indices:
            completion_metadata[index] = json.loads(line)["prompt_metadata"]
            if len(completion_metadata) == len(target_indices):
                break

missing_indices = target_indices - completion_metadata.keys()
if missing_indices:
    raise ValueError(f"Missing completion metadata for sample indices: {sorted(missing_indices)}")


def flatten_scalar_metadata(metadata, prefix=""):
    """Return dotted paths for scalar values in a nested metadata dictionary."""
    flattened = {}
    for key, value in metadata.items():
        path = f"{prefix}.{key}" if prefix else key
        if isinstance(value, dict):
            flattened.update(flatten_scalar_metadata(value, path))
        elif not isinstance(value, (list, tuple, set)):
            flattened[path] = value
    return flattened


metadata_rows = [
    flatten_scalar_metadata(completion_metadata[index])
    for index in sample_indices
]
metadata_df = pd.DataFrame(metadata_rows)
metadata_fields = sorted(metadata_df.columns)

seconds_to_months = 3.80517e-7
unit_to_months = {
    "second": seconds_to_months, "seconds": seconds_to_months,
    "minute": 60 * seconds_to_months, "minutes": 60 * seconds_to_months,
    "hour": 3_600 * seconds_to_months, "hours": 3_600 * seconds_to_months,
    "day": 86_400 * seconds_to_months, "days": 86_400 * seconds_to_months,
    "week": 604_800 * seconds_to_months, "weeks": 604_800 * seconds_to_months,
    "month": 1.0, "months": 1.0, "year": 12.0, "years": 12.0,
    "decade": 120.0, "decades": 120.0,
    "century": 1_200.0, "centuries": 1_200.0,
    "millennium": 12_000.0, "millennia": 12_000.0,
}
time_horizon_months = []
for index in sample_indices:
    metadata = completion_metadata[index]
    value = metadata["base_value"] if "base_value" in metadata else metadata["value"]
    unit = metadata["base_unit"] if "base_unit" in metadata else metadata["unit"]
    time_horizon_months.append(float(value) * unit_to_months[unit.lower()])

df_projs = pd.DataFrame(projs_np, columns=["PC1", "PC2", "PC3"])
df_projs.insert(0, "sample_index", sample_indices)
df_projs = pd.concat([df_projs, metadata_df], axis=1)
df_projs["time_horizon_months"] = time_horizon_months
if (df_projs["time_horizon_months"] <= 0).any():
    raise ValueError("Time horizons must be positive before taking the logarithm.")
df_projs["log10_time_horizon_months"] = np.log10(df_projs["time_horizon_months"])

color_fields = ["log10_time_horizon_months", *metadata_fields]
print(f"Prepared {len(df_projs)} points with {len(metadata_fields)} metadata fields.")

### 6.2 Interactive Plotly explorer

Choose any scalar metadata field for coloring. To filter points, choose a metadata field and select one or more values to keep; use Ctrl/Cmd-click to select multiple values such as `low` and `medium`. The plot updates whenever a control changes.

In [ ]:
import ipywidgets as widgets
import plotly.express as px
from IPython.display import clear_output, display
from pandas.api.types import is_bool_dtype, is_numeric_dtype

difficulty_field = next(
    (field for field in metadata_fields if field.split(".")[-1] == "difficulty"),
    None,
)
color_dropdown = widgets.Dropdown(
    options=color_fields,
    value="log10_time_horizon_months",
    description="Color by:\u00a0",
    layout=widgets.Layout(width="500px"),
)
filter_dropdown = widgets.Dropdown(
    options=[("(no filter)", None), *[(field, field) for field in metadata_fields]],
    value=difficulty_field,
    description="Filter by:\u00a0",
    layout=widgets.Layout(width="500px"),
)
filter_values = widgets.SelectMultiple(
    description="Keep:\u00a0",
    rows=6,
    layout=widgets.Layout(width="500px"),
)
plot_output = widgets.Output()


def available_values(field):
    values = df_projs[field].dropna().unique().tolist()
    return sorted(values, key=lambda value: str(value))


def update_filter_values():
    field = filter_dropdown.value
    values = available_values(field) if field is not None else []
    filter_values.options = [(str(value), value) for value in values]
    filter_values.value = tuple(values)
    filter_values.disabled = field is None


def render_plot(change=None):
    filtered = df_projs
    filter_field = filter_dropdown.value
    if filter_field is not None:
        filtered = filtered[filtered[filter_field].isin(filter_values.value)]

    with plot_output:
        clear_output(wait=True)
        if filtered.empty:
            print("No points match the selected filter values.")
            return

        color_field = color_dropdown.value
        plot_data = filtered.copy()
        numeric_color = (
            is_numeric_dtype(plot_data[color_field])
            and not is_bool_dtype(plot_data[color_field])
        )
        if not numeric_color:
            plot_data[color_field] = (
                plot_data[color_field].astype("string").fillna("<missing>")
            )

        hover_fields = ["sample_index", "time_horizon_months"]
        if filter_field is not None and filter_field != color_field:
            hover_fields.append(filter_field)
        fig = px.scatter_3d(
            plot_data,
            x="PC1",
            y="PC2",
            z="PC3",
            color=color_field,
            color_continuous_scale="Viridis" if numeric_color else None,
            hover_data=hover_fields,
            title=(
                f"PCA projections colored by {color_field} "
                f"({len(plot_data):,} of {len(df_projs):,} points)"
            ),
            opacity=0.7,
        )
        fig.update_traces(marker={"size": 4})
        fig.show()


def on_filter_field_change(change):
    update_filter_values()
    render_plot()


color_dropdown.observe(render_plot, names="value")
filter_dropdown.observe(on_filter_field_change, names="value")
filter_values.observe(render_plot, names="value")
update_filter_values()
display(widgets.VBox([color_dropdown, filter_dropdown, filter_values, plot_output]))
render_plot()

In [ ]:
df_projs